In [1]:
!wget --no-check https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/phix.fa

--2026-11-01 08:17:46--  https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/phix.fa
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving d28rh4a8wq0iu5.cloudfront.net (d28rh4a8wq0iu5.cloudfront.net)... 108.156.162.59, 108.156.162.125, 108.156.162.135, ...
Connecting to d28rh4a8wq0iu5.cloudfront.net (d28rh4a8wq0iu5.cloudfront.net)|108.156.162.59|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5528 (5.4K) [application/octet-stream]
Saving to: ‘phix.fa’

phix.fa             100%[===================>]   5.40K  --.-KB/s    in 0s      

2026-11-01 08:17:47 (144 MB/s) - ‘phix.fa’ saved [5528/5528]



In [2]:
def readGenome(filename):
    genome = ''

    with open(filename, 'r') as f:
        for line in f:
            if not line[0] == '>':
                genome += line.rstrip()

    return genome

In [3]:
genome = readGenome("phix.fa")

In [8]:
def naive(pattern, text):
    occurrences = list()

    for i in range(len(text) - len(pattern) + 1):
        match = True
        for j in range(len(pattern)):
            if text[i + j] != pattern[j]:
                match = False
                break
        if match:
            occurrences.append(i)
    return occurrences

In [9]:
t = "AGCTACGAGCAGT"
p = "AG"
naive(p, t)

[0, 7, 10]

In [10]:
t[10:12]

'AG'

In [11]:
import random

In [14]:
def generateReads(genome, numReads, readLength):
    reads = list()
    for _ in range(numReads):
        start = random.randint(0, len(genome) - readLength) - 1
        reads.append(genome[start: start + readLength])
    return reads

In [15]:
reads = generateReads(genome, 100, 100)

numMatched = 0

for read in reads:
    matches = naive(read, genome)
    if len(matches) > 0:
        numMatched += 1

print(f"{numMatched} / {len(reads)} reads matched exactly")

100 / 100 reads matched exactly


In [16]:
!wget --no-check https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/ERR266411_1.first1000.fastq

--2026-11-01 13:53:04--  https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/ERR266411_1.first1000.fastq
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving d28rh4a8wq0iu5.cloudfront.net (d28rh4a8wq0iu5.cloudfront.net)... 99.84.8.60, 99.84.8.87, 99.84.8.221, ...
Connecting to d28rh4a8wq0iu5.cloudfront.net (d28rh4a8wq0iu5.cloudfront.net)|99.84.8.60|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 254384 (248K) [audio/mpeg]
Saving to: ‘ERR266411_1.first1000.fastq’

ERR266411_1.first10 100%[===================>] 248.42K   111KB/s    in 2.2s    

2026-11-01 13:53:07 (111 KB/s) - ‘ERR266411_1.first1000.fastq’ saved [254384/254384]



In [17]:
def readFastq(filename):
    sequences = list()
    qualities = list()
    with open(filename, 'r') as fh:
        while True:
            fh.readline()
            seq = fh.readline().rstrip()
            fh.readline()
            quality = fh.readline().rstrip()

            if len(seq) == 0:
                break
            sequences.append(seq)
            qualities.append(quality)
    return sequences, qualities

In [18]:
phix_reads, _ = readFastq("ERR266411_1.first1000.fastq")

In [20]:
numMatched = 0
n = 0

for read in phix_reads:
    read = read[:30]
    matches = naive(read, genome)
    n += 1
    if len(matches) > 0:
        numMatched += 1

print(f"{numMatched} / {n} reads matched exactly")

459 / 1000 reads matched exactly


In [23]:
def reverseComplement(seq):
    complement = {"A" : "T", "C" : "G", "G" : "C", "T" : "A", "N" : "N"}
    rev_comp = ""

    for base in seq:
        rev_comp = complement[base] + rev_comp

    return rev_comp

In [26]:
numMatched = 0
n = 0

for read in phix_reads:
    read = read[:30]
    matches = naive(read, genome)
    matches.extend(naive(reverseComplement(read), genome))
    n += 1
    if len(matches) > 0:
        numMatched += 1

print(f"{numMatched} / {n} reads matched exactly")

932 / 1000 reads matched exactly
